In [1]:
import os

from openai import OpenAI

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

client = OpenAI(api_key=OPENAI_API_KEY)

completion = client.chat.completions.create(
    model='gpt-3.5-turbo-0125',
    messages=[{'role': 'user', 'content': 'hi'}],
    temperature=0.0,
)

In [2]:
import json

with open('./res/reviews.json', 'r') as f:
    review_list = json.load(f)

review_list[:3]

[{'title': '子供には最高屋上スパ',
  'review': '2家族10人で利用させて頂きました。和室湖畔側のお部屋は景色も良く清潔感あふれるお部屋でした。夕食朝食ともに大満足でした。なにより子供達が屋上スパに大大大満足でした。いつも忙しく子供との時間もとれて無かったので。とても良い旅行になりました。ありがとう感謝',
  'star': 5,
  'date': '2025年12月',
  'room': 5,
  'bath': 5,
  'breakfast': 5,
  'dinner': 5,
  'service': 5,
  'cleanliness': 5},
 {'title': '大変満足してます',
  'review': 'ご飯も美味しくて、温泉もよく、部屋もそれなりでよかった。花火も綺麗にみることができて、満足しています。また利用したいと思います。',
  'star': 4,
  'date': '2026年2月',
  'room': 4,
  'bath': 4,
  'breakfast': 4,
  'dinner': 4,
  'service': 4,
  'cleanliness': 4},
 {'title': '夜明けの天空スパはステキ',
  'review': '16時前にチェックインしたのですが既に18時台のお食事は満席でした。湖が見える良い部屋だったため20時の花火はお部屋で見たかったので、19時のお食事だとせっかくのバイキングも何となく急かされて、途中でお部屋に戻ってからまた食事席に着くのもなぁと…少し残念でした。(まだお食事中って札を置いて席を立つことは可能)夕食も朝食も種類が沢山あって、選ぶのが楽しかったです。天空スパは早朝に入りましたが、段々夜が明けていく感じがステキでした。',
  'star': 5,
  'date': '2026年2月',
  'room': 5,
  'bath': 5,
  'breakfast': 5,
  'dinner': 5,
  'service': 5,
  'cleanliness': 5}]

In [3]:
good_cnt, bad_cnt = 0, 0
for r in review_list:
    if r['star'] == 5:
        good_cnt += 1
    else:
        bad_cnt += 1

print(f'高評価（5点）: {good_cnt}件, それ以外: {bad_cnt}件')

高評価（5点）: 53件, それ以外: 97件


In [4]:
reviews_good, reviews_bad = [], []

for r in review_list:
    if r['star'] == 5:
        reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
    else:
        reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

reviews_good_text = '\n'.join(reviews_good)
reviews_bad_text = '\n'.join(reviews_bad)

In [5]:
import datetime
import re

def preprocess_reviews(path='./res/reviews.json'):
    with open(path, 'r', encoding='utf-8') as f:
        review_list = json.load(f)

    reviews_good, reviews_bad = [], []

    current_date = datetime.datetime.now()
    date_boundary = current_date - datetime.timedelta(days=6*30)

    for r in review_list:
        review_date_str = r.get('date', '')
        # 「YYYY年MM月」形式の日付をパース
        m = re.match(r'(\d{4})年(\d{1,2})月', review_date_str)
        if m:
            review_date = datetime.datetime(int(m.group(1)), int(m.group(2)), 1)
        else:
            review_date = current_date

        if review_date < date_boundary:
            continue

        if r['star'] == 5:
            reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
        else:
            reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

    reviews_good_text = '\n'.join(reviews_good)
    reviews_bad_text = '\n'.join(reviews_bad)

    return reviews_good_text, reviews_bad_text

good, bad = preprocess_reviews()

In [ ]:
def pairwise_eval(reviews, answer_a, answer_b):
    eval_prompt = f"""[System]
Please act as an impartial judge and evaluate the quality of the Japanese summaries provided by two
AI assistants to the set of user reviews on accommodations displayed below. You should choose the assistant that
follows the user’s instructions and answers the user’s question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.
[User Reviews]
{reviews}
[The Start of Assistant A’s Answer]
{answer_a}
[The End of Assistant A’s Answer]
[The Start of Assistant B’s Answer]
{answer_b}
[The End of Assistant B’s Answer]"""
    
    completion = client.chat.completions.create(
        model='gpt-4o-2024-05-13',
        messages=[{'role': 'user', 'content': eval_prompt}],
        temperature=0.0
    )

    return completion

In [7]:
PROMPT_BASELINE = f"""以下の宿泊施設のレビューを5文以内で要約してください："""

In [8]:
reviews, _ = preprocess_reviews(path='./res/reviews.json')

def summarize(reviews, prompt, temperature=0.0, model='gpt-3.5-turbo-0125'):
    prompt = prompt + '\n\n' + reviews

    completion = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature
    )

    return completion

print(summarize(reviews, PROMPT_BASELINE).choices[0].message.content)

1. 和室湖畔側のお部屋で家族10人で宿泊。夕食朝食も大満足。子供達は屋上スパに大満足。
2. 湖が見える部屋で18時台の食事が満席で残念。天空スパは早朝に入り、夜が明ける感じがステキ。
3. 急な骨折で和室を希望し、対応してもらい感謝。眺めの良いお部屋で静養。スタッフの対応に感謝。
4. 露天風呂が素晴らしかった。朝早く湖や山なみが見えて幻想的。
5. 冬の阿寒湖を楽しんだ。温泉、サウナ、料理全て満足。アップグレードしてもらい感謝。


In [ ]:
reviews_good_all, reviews_bad_all = preprocess_reviews()
good_list = reviews_good_all.split('\n') if reviews_good_all else []
bad_list = reviews_bad_all.split('\n') if reviews_bad_all else []
reviews_for_real = '\n'.join(good_list[:20] + bad_list[:10])

summary_real = summarize(reviews_for_real, prompt_improved, temperature=0.0, model='gpt-4o-2024-05-13').choices[0].message.content
print(summary_real)

In [14]:
print(pairwise_eval(reviews, summarize(reviews, PROMPT_BASELINE).choices[0].message.content, summary_real).choices[0].message.content)

APITimeoutError: Request timed out.

In [11]:
eval_count = 10

summaries_baseline = [summarize(reviews, PROMPT_BASELINE, temperature=1.0).choices[0].message.content for _ in range(eval_count)]
summaries_baseline

['1. 湖畔側の和室で清潔で景色も良く、夕食・朝食も大満足。子供たちは屋上スパに大満足だった。\n2. 湖が見える部屋で18時の食事が満席だったが、20時の花火を見るため急かされることも。天空スパは早朝がおすすめ。\n3. 急な骨折でも親切に対応し、静養ができた。眺めの良い部屋で花火や湖が楽しめ、スタッフの対応も素晴らしかった。\n4. 食事も美味しく、露天風呂は朝早く湖や山を見ながら幻想的な時間を過ごせた。\n5. 湖凍っている風景や温泉の良さに大満足。料理の品数も豊富で、また訪れたいと思う宿。',
 '1. 2家族10人で利用。和室湖畔側のお部屋は清潔感あり、夕食朝食も大満足。子供たちは屋上スパに大満足で良い旅行だった。\n2. 夜中の食事が満席だったため、時間に余裕が欲しかった。バイキングは楽しく、天空スパでの体験が素晴らしかった。\n3. 骨折で急なお願いにも快く対応してもらい、和室を提供してもらい感謝。美しい景色を眺めながら静養でき、スタッフの対応も満足。\n4. 露天風呂が素晴らしく、湖や山の幻想的な景色を楽しめた。食事も美味しく満足できる宿。\n5. 食事が豊富で美味しく、露天風呂も広々していて良いお風呂。屋上の施設も最高で、スタッフの対応も良かった。',
 '1. 和室湖畔側のお部屋で10人のグループで利用。夕食朝食ともに満足。子供が屋上スパで大満足だった。\n2. バイキング形式の食事が豊富で美味しかったが、食事の時間が残念。天空スパで湖の景色を楽しめた。\n3. 怪我で静養のため急な予約したが、親切に対応。眺め良いお部屋で回復の時間を過ごすことができた。\n4. 露天風呂が素晴らしかった。朝早く行って湖や山などを眺め、幻想的な時間を過ごすことができた。\n5. 料理の品数が豊富で大満足。スパも楽しめ、また季節を変えて訪れたいと思える宿。',
 '1. 家族10人で和室湖畔側のお部屋を利用。夕食朝食も大満足。子供達は屋上スパに大満足。\n2.18時台のお食事満席で残念な体験。朝食時の山並みの景色は感動的。\n3.急な骨折のため安静にする必要があったが、対応が素晴らしかった。眺望の良いお部屋で穏やかな滞在。\n4.露天風呂が素晴らしい。朝早くからの湖や山の景色が幻想的。\n5.冬の阿寒湖を楽しむ形で訪れ、料理や温泉、サウナの充実さに感動。次は違う季

In [15]:
from tqdm import tqdm

def pairwise_eval_batch(reviews, answers_a, answers_b):
    a_cnt, b_cnt, draw_cnt = 0, 0, 0
    for i in tqdm(range(len(answers_a))):
        completion = pairwise_eval(reviews, answers_a[i], answers_b[i])
        verdict_text = completion.choices[0].message.content

        if '[[A]]' in verdict_text:
            a_cnt += 1
        elif '[[B]]' in verdict_text:
            b_cnt += 1
        elif '[[C]]' in verdict_text:
            draw_cnt += 1
        else:
            print('Evaluation Error')

    return a_cnt, b_cnt, draw_cnt

wins, losses, ties = pairwise_eval_batch(reviews, summaries_baseline, [summary_real for _ in range(len(summaries_baseline))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [01:26<00:00,  8.65s/it]

Wins: 3, Losses: 7, Ties: 0


In [18]:
prompt = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。

要約結果は以下の条件を満たす必要があります：
1. すべての文は必ず丁寧語（です・ます調）で終わること。
2. 宿泊施設を紹介するトーン＆マナーで作成してください。
  2-1. 良い例
    a) 全体的に良い宿泊施設で、防音も問題なかったという評価です。
    b) 再訪予定だという声が多く見られます。
  2-2. 悪い例
    a) 良い宿泊施設で、防音も問題ありませんでした。
    b) 再訪予定です。
3. 要約結果は最低2文、最大5文で作成してください。
    
以下の宿泊施設のレビューを要約してください："""

eval_count = 10
summaries = [summarize(reviews, prompt, temperature=1.0).choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_real for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [02:35<00:00, 15.59s/it]

Wins: 0, Losses: 10, Ties: 0


In [27]:
import datetime
from dateutil import parser

def preprocess_reviews(path='./res/reviews.json'):
    with open(path, 'r', encoding='utf-8') as f:
        review_list = json.load(f)

    reviews_good, reviews_bad = [], []

    current_date = datetime.datetime.now()
    date_boundary = current_date - datetime.timedelta(days=6*30)

    filtered_cnt = 0
    for r in review_list:
        review_date_str = r['date']
        try:
            review_date = parser.parse(review_date_str)
        except (ValueError, TypeError):
            review_date = current_date

        if review_date < date_boundary:
            continue
        if len(r['review']) < 30:
            filtered_cnt += 1
            continue

        if r['star'] == 5:
            reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
        else:
            reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

    reviews_good = reviews_good[:min(len(reviews_good), 30)]
    reviews_bad = reviews_bad[:min(len(reviews_bad), 30)]

    reviews_good_text = '\n'.join(reviews_good)
    reviews_bad_text = '\n'.join(reviews_bad)

    return reviews_good_text, reviews_bad_text

reviews, _ = preprocess_reviews()

In [20]:
eval_count = 10
summaries = [summarize(reviews, prompt, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_real for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [01:51<00:00, 11.11s/it]

Wins: 2, Losses: 8, Ties: 0


In [24]:
reviews_1shot, _ = preprocess_reviews('./res/kunosato.json')
summary_1shot = summarize(reviews_1shot, prompt, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content
prompt_1shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。

要約結果は以下の条件を満たす必要があります：
1. すべての文は必ず丁寧語（です・ます調）で終わること。
2. 宿泊施設を紹介するトーン＆マナーで作成してください。
  2-1. 良い例
    a) 全体的に良い宿泊施設で、防音も問題なかったという評価です。
    b) 再訪予定だという声が多く見られます。
  2-2. 悪い例
    a) 良い宿泊施設で、防音も問題ありませんでした。
    b) 再訪予定です。
3. 要約結果は最低2文、最大5文で作成してください。

以下はレビューと要約の例です。
例のレビュー：
{reviews_1shot}
例の要約結果：
{summary_1shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_real for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [01:15<00:00,  7.50s/it]

Wins: 0, Losses: 10, Ties: 0


In [25]:
prompt_1shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。以下はレビューと要約の例です。
例のレビュー：
{reviews_1shot}
例の要約結果：
{summary_1shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt_1shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_real for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [01:12<00:00,  7.21s/it]

Wins: 0, Losses: 10, Ties: 0


In [28]:
reviews_1shot, _ = preprocess_reviews('./res/kunosato.json')
summary_1shot = summarize(reviews_1shot, prompt, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content

In [29]:
reviews_2shot, _ = preprocess_reviews('./res/global_view.json')
summary_2shot = summarize(reviews_2shot, prompt_1shot, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content

prompt_2shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。以下はレビューと要約の例です。

例のレビュー 1：
{reviews_1shot}
例の要約結果 1：
{summary_1shot}

例のレビュー 2：
{reviews_2shot}
例の要約結果 2：
{summary_2shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt_2shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_real for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:44<00:00,  4.43s/it]

Wins: 0, Losses: 10, Ties: 0


In [ ]:
prompt_improved = f"""あなたは宿泊施設レビューの要約専門家です。複数のユーザーレビューを読み、宿泊施設全体の特徴を総合的に要約してください。

要約結果は以下の条件を必ず守ってください：
1. 個別のレビューを一つずつ要約するのではなく、全レビューを横断的に分析し、共通する意見や傾向を統合した総合要約を作成すること。
2. レビューに繰り返し登場する施設名・料理名・サービス名などの固有名詞は具体的に記載すること（例：「天空ガーデンスパ」「ジンギスカン」「海鮮丼」など）。
3. 良い点だけでなく、複数のレビューで指摘されている改善点や不満点も1文程度含めること。
4. すべての文は丁寧語（です・ます調）で終わること。
5. 宿泊施設を紹介するトーン＆マナーで、最低2文、最大5文で作成すること。

以下の宿泊施設のレビューを要約してください："""

eval_count = 10
summaries_improved = [summarize(reviews, prompt_improved, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries_improved, [summary_real for _ in range(len(summaries_improved))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

In [ ]:
# good + bad レビューを両方渡して要約（合計30件に制限）
reviews_good, reviews_bad = preprocess_reviews()
good_list = reviews_good.split('\n') if reviews_good else []
bad_list = reviews_bad.split('\n') if reviews_bad else []
reviews_all = '\n'.join(good_list[:20] + bad_list[:10])

eval_count = 10
summaries_improved = [summarize(reviews_all, prompt_improved, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews_all, summaries_improved, [summary_real for _ in range(len(summaries_improved))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')